# 11 · Recipes

A recipe is your processing chain (steps and their settings) saved as a small JSON file, so every new file is processed exactly the same way.

```python
recipe = stk.Recipe.from_history(processed)          # from what you did
recipe = stk.Recipe().despike().trim(200, 1750).baseline().smooth().normalize()   # or step by step
recipe.save('recipes/sers_dyes')                     # -> recipes/sers_dyes.json
new = stk.apply('new_sample.xlsx', 'recipes/sers_dyes.json')   # load + process in one line
```

In [ ]:
import matplotlib.pyplot as plt
import spectrum_toolkit as stk
from spectrum_toolkit import Recipe

XL = 'Raman shift (cm⁻¹)'
cv = stk.load('../data/CV10-4M.xlsx', verbose=False)

## 1. Process one data set the way you like, then turn it into a recipe

In [ ]:
done = cv.despike(verbose=False).trim(200, 1750).baseline('airpls').smooth('savgol', window=11, order=3).normalize('snv')
recipe = Recipe.from_history(done)
recipe.name = 'SERS dyes, 10-4 M'
print(recipe)

## 2. Save it, look at the file, load it back

In [ ]:
path = recipe.save('../data/recipes/sers_dyes')      # data/ is not in git; keep personal recipes there or anywhere
print(path.read_text())
same = Recipe.load(path)
same == recipe

## 3. Apply it to other files in one line

In [ ]:
mb = stk.apply('../data/MB 10-4M.xlsx', path)
rhb = stk.apply('../data/RhB10-4M.xlsx', path)
for name, d in [('CV', done), ('MB', mb), ('RhB', rhb)]:
    stk.plot(d, kind='mean', labels=[name], xlabel=XL, style='publication', title=f'{name} after the recipe')

## 4. See what every step did

In [ ]:
steps = recipe.run(cv[3])                            # spectrum 4 (has cosmic rays)
for (_, before), (name, after) in zip(steps, steps[1:]):
    if name != 'trim':
        stk.compare(before, after, xlabel=XL)

## 5. Build one by hand

Settings are checked straight away, so a typo fails here rather than halfway through a batch.

In [ ]:
mine = Recipe(name='quick look').despike().baseline('asls', lam=1e5).smooth().normalize('max')
print(mine)
try:
    Recipe().baseline('airpls', lamda=1e6)
except ValueError as e:
    print('Error:', e)